# Pattern: Routing

**Phase 04** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-04/04-routing.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 04-04: Pattern: Routing
Phase 04: Agents - Patterns That Survive Production

A customer support router that:
  1. Classifies intent using a cheap/fast model (claude-3-5-haiku)
  2. Dispatches to the right handler (different system prompts, different models)

Intent categories: simple_faq | account_issue | complaint | escalate | technical
"""

import anthropic
from typing import Callable

client = anthropic.Anthropic()

### Classification

In [ ]:
VALID_INTENTS = ["simple_faq", "account_issue", "complaint", "escalate", "technical"]

CLASSIFICATION_SYSTEM = """You are a customer support message classifier.
Classify the user's message into exactly one of these categories:

- simple_faq: General questions about the product, pricing, hours, features, policies
- account_issue: Questions about the user's specific account, billing, subscription, or data
- complaint: Expressions of frustration, negative experience, or request for compensation
- escalate: Threats of legal action, regulatory complaints, or requests to speak to a manager
- technical: Bug reports, error messages, integration issues, or technical troubleshooting

Examples:
- "What are your business hours?" -> simple_faq
- "I've been charged twice this month" -> account_issue
- "I've been charged twice and I'm furious, this is unacceptable" -> complaint
- "I'm contacting my bank and filing a complaint with the FTC" -> escalate
- "I'm getting a 502 error when I try to log in via SSO" -> technical

Respond with ONLY the category name, nothing else. No explanation, no punctuation."""


def classify_intent(message: str) -> str:
    """
    Classify a customer message into one of 5 intent categories.
    Uses the fast/cheap model. Returns a lowercase intent string.
    Validates against known intents and falls back to 'simple_faq' on parse error.
    """
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=16,
        system=CLASSIFICATION_SYSTEM,
        messages=[{"role": "user", "content": message}]
    )

    intent = response.content[0].text.strip().lower().rstrip(".")

    if intent in VALID_INTENTS:
        return intent

    # Fuzzy match: check if any valid intent appears in the response
    for valid in VALID_INTENTS:
        if valid in intent:
            return valid

    return "simple_faq"  # safe fallback

### Handler system prompts

In [ ]:
FAQ_SYSTEM = """You are a concise customer support assistant for a SaaS product.
Answer only what is asked. Keep responses under 3 sentences.
If you don't have specific information, say so briefly and offer to connect the customer with a human support agent.
Do not speculate or invent policy details."""

TECHNICAL_SYSTEM = """You are a technical support specialist for a SaaS product.
Provide step-by-step troubleshooting instructions.
If the user has not provided an error message, ask for it.
Confirm whether the issue is user-side (configuration) or service-side (bug).
Escalate to engineering if you identify a confirmed service bug."""

ACCOUNT_SYSTEM = """You are an account support specialist for a SaaS product.
You help with billing questions, subscription changes, account access, and data requests.
Always verify the user's identity before discussing specific account details.
Be precise about what actions are possible and what requires manual review.
Do not make promises about specific outcomes without confirming with the back-end team."""

COMPLAINT_SYSTEM = """You are a senior customer relations specialist.
Lead with empathy: acknowledge the customer's frustration directly and specifically.
Do not start with apologies alone - propose a concrete next step.
Validate their experience before explaining what happened or what will change.
Do not promise outcomes you cannot guarantee.
If the issue involves a financial impact, acknowledge it explicitly."""

### Handler functions

In [ ]:
def handle_simple_faq(message: str) -> tuple[str, str]:
    """Handle general FAQ questions. Fast, cheap model."""
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=256,
        system=FAQ_SYSTEM,
        messages=[{"role": "user", "content": message}]
    )
    return response.content[0].text, "claude-3-5-haiku-20241022"


def handle_technical(message: str) -> tuple[str, str]:
    """Handle technical issues and bug reports. Fast model, higher max_tokens."""
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=512,
        system=TECHNICAL_SYSTEM,
        messages=[{"role": "user", "content": message}]
    )
    return response.content[0].text, "claude-3-5-haiku-20241022"


def handle_account_issue(message: str) -> tuple[str, str]:
    """Handle account and billing issues. More capable model for accuracy."""
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=512,
        system=ACCOUNT_SYSTEM,
        messages=[{"role": "user", "content": message}]
    )
    return response.content[0].text, "claude-3-5-sonnet-20241022"


def handle_complaint(message: str) -> tuple[str, str]:
    """Handle complaints and negative experiences. Empathy-first prompt."""
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=512,
        system=COMPLAINT_SYSTEM,
        messages=[{"role": "user", "content": message}]
    )
    return response.content[0].text, "claude-3-5-sonnet-20241022"


def handle_escalate(message: str) -> tuple[str, str]:
    """
    Handle escalation requests. In production: open a high-priority ticket,
    page on-call, or transfer to a human agent. This stub returns a transfer message.
    """
    response = (
        "I'm connecting you with a senior member of our team right away. "
        "You'll receive a personal response within 2 business hours. "
        "Your case reference is #[TICKET_ID]. We take your concerns seriously "
        "and will follow up directly."
    )
    return response, "no-model (escalation to human)"

### Raw dispatch function

In [ ]:
HANDLER_REGISTRY: dict[str, Callable[[str], tuple[str, str]]] = {
    "simple_faq":    handle_simple_faq,
    "technical":     handle_technical,
    "account_issue": handle_account_issue,
    "complaint":     handle_complaint,
    "escalate":      handle_escalate,
}


def route_message(message: str, verbose: bool = True) -> dict:
    """
    Classify a customer message and dispatch to the appropriate handler.
    Returns a dict with: intent, response, model_used.
    """
    intent = classify_intent(message)

    if verbose:
        print(f"  Classified: {intent}")

    handler = HANDLER_REGISTRY.get(intent, handle_simple_faq)
    response_text, model_used = handler(message)

    return {
        "intent": intent,
        "response": response_text,
        "model_used": model_used,
        "classification_model": "claude-3-5-haiku-20241022"
    }

### Router class with registered handlers and fallback

In [ ]:
class Handler:
    def __init__(self, fn: Callable[[str], tuple[str, str]], description: str = ""):
        self.fn = fn
        self.description = description

    def run(self, message: str) -> tuple[str, str]:
        return self.fn(message)


class Router:
    """
    A composable router: classifies input, dispatches to registered handlers.
    Falls back to the fallback handler for unknown or unregistered intents.
    """
    def __init__(self, classifier_fn: Callable[[str], str]):
        self._classifier = classifier_fn
        self._handlers: dict[str, Handler] = {}
        self._fallback: Handler | None = None

    def register(self, intent: str, fn: Callable, description: str = "") -> "Router":
        self._handlers[intent] = Handler(fn, description)
        return self

    def set_fallback(self, fn: Callable, description: str = "") -> "Router":
        self._fallback = Handler(fn, description)
        return self

    def route(self, message: str) -> dict:
        intent = self._classifier(message)
        handler = self._handlers.get(intent)

        if handler is None:
            if self._fallback is not None:
                handler = self._fallback
                resolved_intent = f"fallback (classified: {intent})"
            else:
                return {
                    "intent": intent,
                    "error": f"No handler registered for '{intent}' and no fallback set."
                }
        else:
            resolved_intent = intent

        response_text, model_used = handler.run(message)
        return {
            "intent": resolved_intent,
            "response": response_text,
            "model_used": model_used
        }

    def list_handlers(self) -> None:
        print("Registered handlers:")
        for intent, handler in self._handlers.items():
            print(f"  {intent:<20} {handler.description}")
        if self._fallback:
            print(f"  {'(fallback)':<20} {self._fallback.description}")

### Demo

In [ ]:
TEST_MESSAGES = [
    ("What are your business hours?",
     "simple_faq"),
    ("I'm getting a 502 error when I try to log in via SSO with Okta.",
     "technical"),
    ("I was charged twice for my subscription last month. Can you look into my account?",
     "account_issue"),
    ("This is the third time my data has been corrupted. I'm extremely disappointed and I want a refund.",
     "complaint"),
    ("I'm done waiting. I'm contacting my lawyer and filing a complaint with the CFPB.",
     "escalate"),
]

### Demo

In [ ]:
# Build the router
support_router = (
    Router(classify_intent)
    .register("simple_faq",    handle_simple_faq,    "Fast FAQ - Haiku model")
    .register("technical",     handle_technical,     "Technical support - Haiku model")
    .register("account_issue", handle_account_issue, "Account handling - Sonnet model")
    .register("complaint",     handle_complaint,     "Complaint resolution - Sonnet model")
    .register("escalate",      handle_escalate,      "Human escalation - no model")
    .set_fallback(handle_simple_faq,                  "Default: treat as FAQ")
)

print("Router configuration:")
support_router.list_handlers()
print()

print("=" * 65)
print("Routing test messages")
print("=" * 65)

for message, expected_intent in TEST_MESSAGES:
    print(f"\nMessage: \"{message[:60]}{'...' if len(message) > 60 else ''}\"")
    print(f"Expected intent: {expected_intent}")

    result = support_router.route(message)
    actual_intent = result.get("intent", "error")
    match = "MATCH" if expected_intent in actual_intent else "MISMATCH"

    print(f"Actual intent:   {actual_intent} [{match}]")
    print(f"Model used:      {result.get('model_used', 'N/A')}")
    print(f"Response:        {result.get('response', result.get('error', ''))[:120]}...")

---

## Self-Check

Answer these without running code first:

**1. What is the root cause and the minimal fix?**

- A. The account handler's system prompt needs to be more empathetic.
- B. The classification prompt lacks examples that distinguish account questions (procedural) from complaints (emotional + account reference). Adding 1-2 boundary-case examples to the classification prompt would prevent this misclassification.
- C. The router should always route messages containing 'charged' to the complaint handler.
- D. The account_issue and complaint handlers should be merged into one handler that adapts its tone.

**2. What happens, and what prevents this from being a silent failure?**

- A. The router raises a KeyError exception, which crashes the service.
- B. The router returns the escalation handler's response, since escalation is the last registered handler.
- C. If a fallback handler is registered, the router uses it and records the resolved intent as 'fallback (classified: product_feedback)'. Without a fallback, the router returns an error dict with the unhandled intent.
- D. The router silently drops the message and returns an empty response.

**3. What is the correct model assignment strategy, and what is the approximate cost savings vs. routing everything to the expensive model?**

- A. Use the expensive model for all intents to avoid misrouting costs.
- B. Use the cheapest model for all intents and accept lower quality on litigation_research.
- C. Use the cheap model for general_query (40%) and the expensive model for contract_review, compliance_check, and litigation_research. Savings: approximately 40% * (expensive_cost - cheap_cost) per message.
- D. Use the cheap model for general_query and contract_review (70%), the mid model for compliance_check (20%), and the expensive model for litigation_research (10%). Savings are proportional to the fraction routed to cheaper models.

**4. What is the correct fix in the classify_intent function?**

- A. Add 'account_issue.' as a registered intent that maps to the same handler as 'account_issue'.
- B. Strip trailing punctuation from the model's response before validating against VALID_INTENTS.
- C. Set max_tokens to 1 to prevent the model from generating punctuation.
- D. Switch to a different model that does not add punctuation.

**5. What is the most efficient next step?**

- A. Re-label the 7 misclassified messages as account_issue since the model might be right.
- B. Add 5-8 examples to the classification prompt that show the boundary between complaint and account_issue, focusing on messages that contain both emotional language and account references.
- C. Switch the classifier to a more powerful model to improve overall accuracy.
- D. Merge the complaint and account_issue handlers into one handler and collapse the two intents.

**6. What is the likely cause and the correct diagnosis approach?**

- A. The complaint handler is now routing to a more expensive model due to a config change.
- B. A longer system prompt increases input token count, which increases cost but should not significantly affect latency.
- C. The system prompt growth increases input tokens on every call, which increases both cost and latency. Profile the token count before and after the prompt growth, and reduce the prompt to the minimal effective set of instructions.
- D. The latency increase is due to increased production traffic, not the system prompt.

**7. **

- A. Add logging to the classification prompt so the model knows to mark escalation messages.
- B. Add the compliance logging call inside the handle_escalate function, before returning the response.
- C. Add logging in the router's route() method, after the handler returns, for all intents.
- D. Add logging in a separate post-processing step that runs asynchronously after the route() call returns.

**8. What is the minimum number of model tiers you need, and how should you assign them?**

- A. One tier: use the same model for all intents to simplify the system.
- B. Two tiers: a cheap/fast model for documentation (50%) and a more capable model for API debugging, billing, and feature requests (50%).
- C. Four tiers: a different model for each intent.
- D. Two tiers: a cheap model for documentation and feature requests (55%), and a capable model for API debugging and billing (45%).

_Answers are in `checks.json` in the lesson directory._